# CosMx data loading, QC filtering, and metadata merging

**Goal:** Load raw CosMx NanoString data from four TMA slides, assign globally unique FOV
identifiers across slides, merge TMA spatial metadata, apply cell-level QC filters,
normalize counts, and save a single merged AnnData object for all downstream CosMx analyses.

**Input:** Raw CosMx expression matrices, metadata files, and FOV position files for
slides LungAdCa20141–20144; TMA spatial metadata CSV mapping FOV to patient/region.

**Output:** `outputs/processed/combined_adata_qc_filtered.h5ad` — merged, QC-filtered,
normalized AnnData ready for UMAP embedding and cell typing.

## Setup

In [ ]:
# standard libraries
import yaml
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell and spatial analysis
import anndata
from anndata import AnnData
import scanpy as sc
import squidpy as sq
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF — required by most journals
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in CosMx/scRNA UMAPs

In [ ]:
# load paths from central config — update paths_config.yml for your machine, not this file
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# raw CosMx slide directories
cosmx_raw_dir = Path(cfg['datasets']['cosmx']['raw'])

# TMA spatial metadata (FOV → patient/region mapping)
tma_metadata_path = Path(cfg['datasets']['cosmx']['metadata'])

# output paths — resolved against repo root
processed_out = repo_root / cfg['outputs']['processed']
processed_out.mkdir(parents=True, exist_ok=True)

## Load raw CosMx slides

Each of the four TMA slides (LungAdCa20141–20144) is loaded independently using
`squidpy.read.nanostring`. After loading, a globally unique `fov_shifted` identifier
is computed for each slide by offsetting raw FOV numbers by the cumulative FOV count
of all preceding slides. This ensures FOV IDs are non-overlapping across slides when
the objects are concatenated.

In [ ]:
import scipy.io
import scipy.sparse as sp

def load_slide_from_mtx(slide_name, fov_offset, extracted_dir):
    """Load a CosMx slide from extracted MTX files."""
    slide_dir = extracted_dir / slide_name
    
    # load count matrix, genes, barcodes
    counts = scipy.io.mmread(slide_dir / f'{slide_name}_counts.mtx').T.tocsr()
    genes = pd.read_csv(slide_dir / f'{slide_name}_genes.csv')['gene'].values
    meta = pd.read_csv(slide_dir / f'{slide_name}_metadata_file.csv', index_col=0)
    
    # build AnnData
    adata_slide = sc.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
    adata_slide.obs['fov_shifted'] = adata_slide.obs['fov'].astype(int) + fov_offset
    
    return adata_slide

extracted_dir = Path('/scratch/gh8sj/CosMx/extracted_csvs')

slides = [
    ('LungAdCa20141', 0),
    ('LungAdCa20142', 273),
    ('LungAdCa20143', 273 + 282),
    ('LungAdCa20144', 273 + 282 + 270),
]

adata_slides = []
for slide_name, fov_offset in slides:
    adata_slide = load_slide_from_mtx(slide_name, fov_offset, extracted_dir)
    adata_slides.append(adata_slide)
    print(f"{slide_name}: {adata_slide.n_obs:,} cells, FOV offset={fov_offset}")

## Concatenate slides

In [ ]:
# concatenate all four slides into a single AnnData
# outer join preserves all genes present in any slide; uns_merge='unique' keeps non-conflicting metadata
adata = anndata.concat(
    adata_slides,
    join='outer',
    label='batch',
    keys=[s[0] for s in slides],
    uns_merge='unique',
)
print(f"Combined: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
adata

## Merge TMA spatial metadata

The TMA metadata CSV maps each `fov_shifted` value to its associated patient ID and
tissue region (tumor or tonsil control). FOVs with no region annotation are control
tonsil cores and are labeled accordingly.

In [ ]:
# load TMA metadata — maps fov_shifted to patient ID and tissue region
metadata = pd.read_csv(tma_metadata_path, index_col=0)

# FOVs with no region annotation are tonsil control cores
metadata['region'] = metadata['region'].fillna('tonsil')

# merge metadata into obs on fov_shifted; drop the redundant fov_y column produced by the merge
adata.obs = pd.merge(adata.obs, metadata, on='fov_shifted', suffixes=(None, '_y'), copy=False)
if 'fov_y' in adata.obs.columns:
    adata.obs.drop(columns=['fov_y'], inplace=True)

print(f"Region counts after merge:")
print(adata.obs['region'].value_counts())

## QC filtering

Cells are filtered on total transcript count and the NanoString QC flag. The count
window of 20–2000 transcripts per cell follows NanoString's recommended thresholds
for the CosMx 1000-plex panel. Cells flagged by the instrument's internal QC
pipeline (`qcCellsFlagged`) are also removed.

In [ ]:
n_before = adata.n_obs

# filter on transcript count window (NanoString recommended: min 20, max 2000)
sc.pp.filter_cells(adata, min_counts=20)
sc.pp.filter_cells(adata, max_counts=2000)

# remove cells flagged by NanoString's internal QC pipeline
adata = adata[adata.obs['qcCellsFlagged'] == False].copy()

print(f"Cells before QC: {n_before:,}")
print(f"Cells after QC:  {adata.n_obs:,} ({n_before - adata.n_obs:,} removed)")

## Normalization and preprocessing

Raw counts are stored in `adata.layers['counts']` before normalization to allow
downstream tools that require integer counts (e.g., DESeq2 via PyDESeq2) to access
the originals. Cells are normalized to the same total count, log1p-transformed,
and a PCA + neighbor graph is computed in preparation for UMAP embedding
(run in the dedicated `umap_cosmx` notebook).

In [ ]:
%%time

# preserve raw counts before normalization — required for DESeq2 downstream
adata.layers['counts'] = adata.X.copy()

# normalize, log-transform, and compute PCA + neighbor graph
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)

print("Preprocessing complete.")
adata

## Save

The merged, QC-filtered, and preprocessed AnnData is saved to `outputs/processed/`.
This file is the entry point for all downstream CosMx notebooks.

In [ ]:
out_path = processed_out / 'combined_adata_qc_filtered.h5ad'
adata.write(out_path)
print(f"Saved {adata.n_obs:,} cells → {out_path}")